**RAG Based Wikipedia Knowledge Base Search Article**
---



In [1]:
!pip3 install wikipedia pandas tqdm

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=c3bb9273d3a0ed94e4218c915d2cc7c04e506a5ce7c8061ba39f2260acd71f7d
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =====================================================
# Configuration
# =====================================================

OUTPUT_DIR = Path("AI_Knowledge_Base")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "AIML-RAG-Dataset-Builder/1.0"
}

SEARCH_LIMIT = 10
MAX_RETRIES = 3
SLEEP_TIME = 1

# =====================================================
# Seed Keywords
# =====================================================

KEYWORDS = [

    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Model",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision",
    "Generative AI",
    "Transformer",
    "Neural Network",
    "Prompt Engineering",
    "Retrieval Augmented Generation",
    "Semantic Search",
    "Vector Database",
    "Feature Engineering"

]

# =====================================================
# Search Wikipedia
# =====================================================

def search_articles(keyword, limit=10):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    titles = []

    for article in data["query"]["search"]:
        titles.append(article["title"])

    return titles

# =====================================================
# Download Wikipedia Article
# =====================================================

def download_article(title):

    url = "https://en.wikipedia.org/w/api.php"

    params = {

        "action": "query",
        "format": "json",
        "titles": title,
        "redirects": 1,
        "prop": "extracts|info",
        "inprop": "url",
        "explaintext": 1

    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    page = list(data["query"]["pages"].values())[0]

    if "missing" in page:
        return None

    return {

        "title": page["title"],
        "text": page.get("extract", ""),
        "url": page.get("fullurl", "")

    }

# =====================================================
# Safe filename
# =====================================================

def clean_filename(name):

    invalid = '<>:"/\\|?*'

    for ch in invalid:
        name = name.replace(ch, "_")

    return name

# =====================================================
# Main Download Loop
# =====================================================

metadata = []
errors = []

visited = set()

print("="*60)
print("Searching Wikipedia")
print("="*60)

candidate_titles = []

for keyword in KEYWORDS:

    print(f"\nSearching : {keyword}")

    try:

        results = search_articles(keyword, SEARCH_LIMIT)

        candidate_titles.extend(results)

        print(f"Found {len(results)} articles")

    except Exception as e:

        print(e)

candidate_titles = sorted(set(candidate_titles))

print("\n")
print("="*60)
print(f"Total Unique Articles Found : {len(candidate_titles)}")
print("="*60)

# =====================================================
# Download
# =====================================================

for title in tqdm(candidate_titles):

    if title in visited:
        continue

    visited.add(title)

    success = False

    for retry in range(MAX_RETRIES):

        try:

            page = download_article(title)

            if page is None:
                raise Exception("Page not found")

            filename = clean_filename(page["title"]) + ".txt"

            filepath = OUTPUT_DIR / filename

            if filepath.exists():

                success = True

                break

            with open(
                filepath,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page["text"])

            metadata.append({

                "Title": page["title"],
                "URL": page["url"],
                "Words": len(page["text"].split()),
                "Characters": len(page["text"]),
                "File": filename

            })

            success = True

            break

        except Exception as e:

            if retry == MAX_RETRIES - 1:

                errors.append({

                    "Title": title,
                    "Error": str(e)

                })

            time.sleep(SLEEP_TIME)

# =====================================================
# Save CSV
# =====================================================

metadata_df = pd.DataFrame(metadata)

metadata_df.to_csv(
    OUTPUT_DIR / "metadata.csv",
    index=False
)

errors_df = pd.DataFrame(errors)

errors_df.to_csv(
    OUTPUT_DIR / "error_log.csv",
    index=False
)

print("\n")
print("="*60)
print("Download Completed")
print("="*60)
print(f"Articles Downloaded : {len(metadata_df)}")
print(f"Failed Downloads    : {len(errors_df)}")
print(f"Folder              : {OUTPUT_DIR}")
print("="*60)

Searching Wikipedia

Searching : Artificial Intelligence
Found 10 articles

Searching : Machine Learning
Found 10 articles

Searching : Deep Learning
Found 10 articles

Searching : Large Language Model
Found 10 articles

Searching : Natural Language Processing
Found 10 articles

Searching : Data Science
Found 10 articles

Searching : Computer Vision
Found 10 articles

Searching : Generative AI
Found 10 articles

Searching : Transformer
Found 10 articles

Searching : Neural Network
Found 10 articles

Searching : Prompt Engineering
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Prompt+Engineering&srlimit=10&format=json

Searching : Retrieval Augmented Generation
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Retrieval+Augmented+Generation&srlimit=10&format=json

Searching : Semantic Search
429 Client Error: Too Many Requests for url: https://en.wikipedia.o

100%|██████████| 89/89 [03:40<00:00,  2.48s/it]



Download Completed
Articles Downloaded : 35
Failed Downloads    : 54
Folder              : AI_Knowledge_Base


**Install Libraries**

In [3]:

!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 101.9 MB/s eta 0:00:00


**Load Wikipedia Text Files**

In [4]:
import os
from langchain_core.documents import Document

documents = []

folder_path = "AI_Knowledge_Base"

for file in os.listdir(folder_path):

    if file.endswith(".txt"):

        with open(
            os.path.join(folder_path, file),
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": file}
                )
            )

print("Documents Loaded:", len(documents))

Documents Loaded: 35


**Split Documents into Chunks**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 2503


**Create Embeddings**

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Create FAISS Vector Store**

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

vectorstore.save_local("wiki_faiss_db")

print("FAISS Database Created")


/tmp/ipykernel_1102/179535320.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS Database Created


**Load Existing Vector Store**

In [8]:
vectorstore = FAISS.load_local(
    "wiki_faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)


**Create Retriever**

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


**Test Retrieval**

In [10]:
query = "What is Machine Learning?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}")
    print(doc.page_content[:500])



Document 1
=== Learning ===
Machine learning is the study of programs that can improve their performance on a given task automatically. It has been a part of AI from the beginning.

There are several kinds of machine learning:

Document 2
=== Learning paradigms ===
Machine learning has involved a variety of approaches to training models, including supervised learning, unsupervised learning, reinforcement learning, and self-supervised learning.

Document 3
The third millennium saw the introduction of systems using machine learning for text classification, such as the IBM Watson. However, experts debate how much "understanding" such systems demonstrate: e.g., according to John Searle, Watson did not even understand the questions.


**Load LLM**

In [13]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto"
)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

**Build Simple RAG Function**

In [14]:
def ask_rag(question):

    docs = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = llm(
        prompt,
        max_new_tokens=200,
        temperature=0.7
    )

    return response[0]["generated_text"]

**Ask Questions**

In [15]:
question = "What is Natural Language Processing?"

answer = ask_rag(question)

print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



    Context:
    Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.


== History ==
Natural Language Processing is a bimonthly peer-reviewed academic journal published by Cambridge University Press which covers research and software in natural language processing. It was established in 1995 as Natural Language Engineering, obtaining its current title in 2024. Other than original publications on theoretical and applied aspects of computational linguistics, the journal also contains Industry Watch and Emerging Trends columns tracking developments in the field. The edit